# TSA-Net Zero-Shot Evaluation on FaceForensics++
Loads the trained `TSANet` checkpoint and evaluates 100 samples from `original` (real) and `Deepfakes` / `Face2Face` / `FaceSwap` / `NeuralTextures` (fake) under `/kaggle/input/datasets/xdxd003`.

## Cell 1: Core Imports & Environment Setup

In [1]:
# ==========================================
# Cell 1: Core Imports & Environment Setup
# ==========================================
import sys
import os
import random
import json
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc, accuracy_score
import warnings

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Environment initialized on {device} | Random seed locked to 42")


✓ Environment initialized on cuda | Random seed locked to 42


## Cell 2: TSA-Net Architecture Definition

In [2]:
# ==========================================
# Cell 2: TSA-Net Architecture Definition
# ==========================================
class TSANet(nn.Module):
    def __init__(self, d_model=256):
        super(TSANet, self).__init__()

        efficientnet = models.efficientnet_b0(weights=None)
        self.v_backbone = efficientnet.features
        self.v_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.v_fc = nn.Linear(1280, d_model)

        self.a_backbone = nn.Sequential(
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(128)
        )

        class CrossAttention(nn.Module):
            def __init__(self, d_model=256, nhead=8):
                super().__init__()
                self.v_to_a_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
                self.a_to_v_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
                self.norm_v = nn.LayerNorm(d_model)
                self.norm_a = nn.LayerNorm(d_model)

            def forward(self, v_feat, a_feat):
                v_attn, _ = self.v_to_a_attn(query=v_feat, key=a_feat, value=a_feat)
                v_out = self.norm_v(v_feat + v_attn)

                a_attn, _ = self.a_to_v_attn(query=a_feat, key=v_feat, value=v_feat)
                a_out = self.norm_a(a_feat + a_attn)

                return v_out, a_out

        self.cross_attn = CrossAttention(d_model=d_model, nhead=8)

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, visual_inputs, audio_inputs):
        batch_size, num_frames, c, h, w = visual_inputs.shape
        v_x = visual_inputs.view(batch_size * num_frames, c, h, w)
        v_x = self.v_backbone(v_x)
        v_x = self.v_pool(v_x).flatten(1)
        v_x = self.v_fc(v_x)

        v_x = F.layer_norm(v_x, (v_x.size(-1),))
        v_feat = v_x.view(batch_size, num_frames, -1)

        a_x = self.a_backbone(audio_inputs)
        a_feat = a_x.transpose(1, 2)
        a_feat = F.layer_norm(a_feat, (a_feat.size(-1),))

        v_attn, a_attn = self.cross_attn(v_feat, a_feat)

        v_pooled = torch.mean(v_attn, dim=1)
        a_pooled = torch.mean(a_attn, dim=1)

        fused = torch.cat([v_pooled, a_pooled], dim=-1)
        logits = self.classifier(fused)

        return logits

model = TSANet(d_model=256).to(device)
print("✓ TSANet structure loaded.")


✓ TSANet structure loaded.


## Cell 3: Load Trained Checkpoint

In [3]:
# ==========================================
# Cell 3: Load Trained Checkpoint
# ==========================================
CHECKPOINT_PATH = "/kaggle/input/notebooks/mugabashuqramefgi/mpfive-data-augmentation-pipeline/tsa_net_augmented_best.pth"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at: {CHECKPOINT_PATH}\n"
        "Make sure the corresponding Kaggle Dataset/Notebook output is attached in the 'Data' panel."
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict, strict=True)
model.eval()
print(f"✓ Successfully loaded checkpoint from: {CHECKPOINT_PATH}")


✓ Successfully loaded checkpoint from: /kaggle/input/notebooks/mugabashuqramefgi/mpfive-data-augmentation-pipeline/tsa_net_augmented_best.pth


## Cell 4: FaceForensics++ File Discovery (100 balanced samples)

In [4]:
# ==========================================
# Cell 4: FaceForensics++ File Discovery
# ==========================================
FFPP_ROOT = "/kaggle/input/datasets/xdxd003"

def discover_ffpp_videos(root, n_total=100, seed=42):
    random.seed(seed)
    video_exts = ('.mp4', '.avi', '.mov', '.mkv')
    categories = {
        'original': 1.0,        # real
        'Deepfakes': 0.0,
        'Face2Face': 0.0,
        'FaceSwap': 0.0,
        'NeuralTextures': 0.0,
    }

    per_cat_files = {cat: [] for cat in categories}
    for dirpath, _, files in os.walk(root):
        for cat in categories:
            if cat.lower() in dirpath.lower():
                for f in files:
                    if f.lower().endswith(video_exts):
                        per_cat_files[cat].append(os.path.join(dirpath, f))

    for cat in per_cat_files:
        print(f"  found {len(per_cat_files[cat])} videos in '{cat}'")

    n_cats = len([c for c in per_cat_files if len(per_cat_files[c]) > 0])
    if n_cats == 0:
        return [], []
    per_cat_quota = max(1, n_total // n_cats)

    paths, labels = [], []
    for cat, label in categories.items():
        files = per_cat_files[cat]
        if not files:
            continue
        random.shuffle(files)
        chosen = files[:per_cat_quota]
        paths.extend(chosen)
        labels.extend([label] * len(chosen))

    combined = list(zip(paths, labels))
    random.shuffle(combined)
    combined = combined[:n_total]
    paths, labels = zip(*combined) if combined else ([], [])
    return list(paths), list(labels)

ffpp_paths, ffpp_labels = discover_ffpp_videos(FFPP_ROOT, n_total=100)
print(f"✓ Selected {len(ffpp_paths)} FF++ samples "
      f"({sum(l == 1.0 for l in ffpp_labels)} real / {sum(l == 0.0 for l in ffpp_labels)} fake)")

if len(ffpp_paths) == 0:
    raise RuntimeError(f"No FF++ videos found under {FFPP_ROOT}. Check the mounted path/folder names.")

print("\nSample paths:")
for p in ffpp_paths[:5]:
    print(" ", p)


  found 1000 videos in 'original'
  found 1000 videos in 'Deepfakes'
  found 1000 videos in 'Face2Face'
  found 1000 videos in 'FaceSwap'
  found 1000 videos in 'NeuralTextures'
✓ Selected 100 FF++ samples (20 real / 80 fake)

Sample paths:
  /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/NeuralTextures/962_929.mp4
  /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes/157_245.mp4
  /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/NeuralTextures/314_347.mp4
  /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes/957_959.mp4
  /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes/795_907.mp4


## Cell 5: Dataset & Preprocessing Pipeline

In [5]:
# ==========================================
# Cell 5: Dataset & Preprocessing Pipeline
# ==========================================
class FFPPDataset(Dataset):
    def __init__(self, file_paths, labels, num_frames=5):
        self.file_paths = file_paths
        self.labels = labels
        self.num_frames = num_frames
        self.audio_failures = 0
        self.video_failures = 0

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=128)

    def __len__(self):
        return len(self.file_paths)

    def _extract_frames_safe(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frames = []
        if total_frames > 0:
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret and frame is not None and frame.size > 0:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(self.transform(frame))
        cap.release()

        if len(frames) == 0:
            self.video_failures += 1
            frames = [torch.zeros((3, 224, 224))] * self.num_frames
        while len(frames) < self.num_frames:
            frames.append(frames[-1])

        return torch.stack(frames[:self.num_frames])

    def _extract_audio_safe(self, video_path):
        try:
            waveform, sample_rate = torchaudio.load(video_path)
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            if sample_rate != 16000:
                waveform = T.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)

            target_len = 16000 * 5
            if waveform.shape[1] > target_len:
                waveform = waveform[:, :target_len]
            elif waveform.shape[1] < target_len:
                waveform = F.pad(waveform, (0, target_len - waveform.shape[1]))

            mel_spec = self.mel_transform(waveform).squeeze(0)
            mel_db = 10.0 * torch.log10(torch.clamp(mel_spec, min=1e-10))

            std_val = mel_db.std()
            if std_val < 1e-5:
                return torch.zeros((128, 128), dtype=torch.float32)

            mel_db = (mel_db - mel_db.mean()) / std_val

            if mel_db.shape[1] < 128:
                mel_db = F.pad(mel_db, (0, 128 - mel_db.shape[1]))
            else:
                mel_db = mel_db[:, :128]

            return mel_db.to(torch.float32)
        except Exception:
            # Handles missing/corrupt audio tracks gracefully
            self.audio_failures += 1
            return torch.zeros((128, 128), dtype=torch.float32)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        target = self.labels[idx]
        v_tensor = self._extract_frames_safe(path)
        a_tensor = self._extract_audio_safe(path)
        return v_tensor, a_tensor, torch.tensor(target, dtype=torch.float32)

ffpp_dataset = FFPPDataset(ffpp_paths, ffpp_labels)
ffpp_loader = DataLoader(ffpp_dataset, batch_size=8, shuffle=False, num_workers=2)
print(f"✓ Dataset & DataLoader ready with {len(ffpp_dataset)} samples.")


✓ Dataset & DataLoader ready with 100 samples.


## Cell 6: Evaluation (Accuracy / ROC-AUC / EER)

In [6]:
# ==========================================
# Cell 6: Evaluation (Accuracy / ROC-AUC / EER)
# ==========================================
def calculate_eer_safe(y_true, y_scores):
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.5, np.array([0, 1]), np.array([0, 1])
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.absolute(fnr - fpr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0
    optimal_threshold = thresholds[eer_idx]
    return eer, optimal_threshold, fpr, tpr

model.eval()
y_true, y_scores, y_preds = [], [], []

print("Running FF++ Zero-Shot Evaluation...")
with torch.no_grad():
    for v_in, a_in, targets in ffpp_loader:
        if v_in.numel() == 0 or a_in.numel() == 0:
            continue
        v_in, a_in = v_in.to(device), a_in.to(device)
        try:
            logits = model(v_in, a_in).squeeze(-1)
        except RuntimeError as e:
            print(f"⚠️ Skipping a batch due to shape error: {e}")
            continue

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        y_true.extend(targets.cpu().numpy().flatten())
        y_scores.extend(probs.cpu().numpy().flatten())
        y_preds.extend(preds.cpu().numpy().flatten())

y_true, y_scores, y_preds = np.array(y_true), np.array(y_scores), np.array(y_preds)

print("\n" + "=" * 60)
print("🔍 FaceForensics++ ZERO-SHOT EVALUATION")
print("=" * 60)
print(f"  • Total Samples Evaluated : {len(y_true)}")
print(f"  • Audio Extraction Errors : {ffpp_dataset.audio_failures} / {len(ffpp_paths)}")
print(f"  • Video Read Failures     : {ffpp_dataset.video_failures} / {len(ffpp_paths)}")
print("=" * 60)

if len(y_true) == 0:
    print("❌ No samples were successfully evaluated.")
else:
    acc = accuracy_score(y_true, y_preds)
    eer, opt_thresh, fpr, tpr = calculate_eer_safe(y_true, y_scores)
    roc_auc = auc(fpr, tpr) if len(np.unique(y_true)) > 1 else 0.5

    print(f"  • Accuracy   : {acc * 100:.2f}%")
    print(f"  • ROC-AUC    : {roc_auc:.4f}")
    print(f"  • EER        : {eer * 100:.2f}%")
    print("=" * 60)


Running FF++ Zero-Shot Evaluation...

🔍 FaceForensics++ ZERO-SHOT EVALUATION
  • Total Samples Evaluated : 100
  • Audio Extraction Errors : 0 / 100
  • Video Read Failures     : 0 / 100
  • Accuracy   : 63.00%
  • ROC-AUC    : 0.5238
  • EER        : 49.38%


In [7]:
# ==============================================================================
# Cell 7: Audio Waveform Energy Diagnostic (Paper Table 1)
# ==============================================================================
import pandas as pd

def check_ffpp_audio_energy(file_paths, labels):
    records = []
    print("🔍 Inspecting raw audio waveform RMS energy across FF++ categories...")
    
    for path, label in zip(file_paths, labels):
        # Determine specific subfolder category
        cat_name = 'Original (Real)'
        for cat in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
            if cat.lower() in path.lower():
                cat_name = cat
                break

        try:
            waveform, sample_rate = torchaudio.load(path)
            rms = torch.sqrt(torch.mean(waveform ** 2)).item()
        except Exception:
            rms = 0.0  # Missing or silent track

        records.append({
            'Category': cat_name,
            'Label': 'Real' if label == 1.0 else 'Fake',
            'Audio_RMS': rms
        })

    df_audio = pd.DataFrame(records)
    
    print("\n" + "=" * 70)
    print(" TABLE 1: AUDIO WAVEFORM ENERGY ANALYSIS ACROSS FF++ CATEGORIES")
    print("=" * 70)
    
    summary_df = df_audio.groupby('Category')['Audio_RMS'].agg(
        Mean_RMS='mean',
        Std_RMS='std',
        Min_RMS='min',
        Max_RMS='max'
    ).reset_index()
    
    print(summary_df.to_string(index=False))
    
    silent_count = (df_audio['Audio_RMS'] < 1e-4).sum()
    total = len(df_audio)
    print(f"\n💡 Audio Degeneracy Summary: {silent_count}/{total} ({silent_count/total*100:.1f}%) "
          f"videos contain near-zero or missing audio signals.")
    print("=" * 70)
    return df_audio

df_audio = check_ffpp_audio_energy(ffpp_paths, ffpp_labels)

🔍 Inspecting raw audio waveform RMS energy across FF++ categories...

 TABLE 1: AUDIO WAVEFORM ENERGY ANALYSIS ACROSS FF++ CATEGORIES
       Category  Mean_RMS  Std_RMS  Min_RMS  Max_RMS
      Deepfakes       0.0      0.0      0.0      0.0
      Face2Face       0.0      0.0      0.0      0.0
       FaceSwap       0.0      0.0      0.0      0.0
 NeuralTextures       0.0      0.0      0.0      0.0
Original (Real)       0.0      0.0      0.0      0.0

💡 Audio Degeneracy Summary: 100/100 (100.0%) videos contain near-zero or missing audio signals.


In [8]:
# ==============================================================================
# Cell 8: Per-Category Manipulation Breakdown (Paper Table 2)
# ==============================================================================
def run_per_category_breakdown(model, dataset, device):
    model.eval()
    records = []
    
    print("🔍 Evaluating TSA-Net performance per specific manipulation method...")
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            v_in, a_in, target = dataset[idx]
            path = dataset.file_paths[idx]
            
            # Identify manipulation type
            cat_name = 'Original (Real)'
            for cat in ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']:
                if cat.lower() in path.lower():
                    cat_name = cat
                    break
            
            v_tensor = v_in.unsqueeze(0).to(device)
            a_tensor = a_in.unsqueeze(0).to(device)
            
            try:
                logits = model(v_tensor, a_tensor).squeeze(-1)
                prob_real = torch.sigmoid(logits).item()
            except Exception as e:
                continue
                
            records.append({
                'Path': path,
                'Category': cat_name,
                'Target': target.item(),
                'Score_Fake': prob_real  # Model output prediction
            })
            
    df = pd.DataFrame(records)
    real_df = df[df['Target'] == 1.0]  # Real baseline
    
    breakdown_rows = []
    fake_categories = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
    
    for cat in fake_categories:
        fake_df = df[df['Category'] == cat]
        if len(fake_df) == 0:
            continue
            
        combined = pd.concat([real_df, fake_df])
        y_true = combined['Target'].values
        y_scores = combined['Score_Fake'].values
        
        y_preds = (y_scores >= 0.5).astype(float)
        acc = accuracy_score(y_true, y_preds) * 100.0
        eer, _, fpr, tpr = calculate_eer_safe(y_true, y_scores)
        roc_auc = auc(fpr, tpr) if len(np.unique(y_true)) > 1 else 0.5
        
        breakdown_rows.append({
            'Manipulation Method': cat,
            'Fake Samples': len(fake_df),
            'Accuracy (%)': f"{acc:.2f}",
            'ROC-AUC': f"{roc_auc:.4f}",
            'EER (%)': f"{eer * 100:.2f}"
        })
        
    print("\n" + "=" * 70)
    print(" TABLE 2: PER-MANIPULATION CATEGORY PERFORMANCE BREAKDOWN")
    print("=" * 70)
    df_breakdown = pd.DataFrame(breakdown_rows)
    print(df_breakdown.to_string(index=False))
    print("=" * 70)
    return df

df_cat_scores = run_per_category_breakdown(model, ffpp_dataset, device)

🔍 Evaluating TSA-Net performance per specific manipulation method...

 TABLE 2: PER-MANIPULATION CATEGORY PERFORMANCE BREAKDOWN
Manipulation Method  Fake Samples Accuracy (%) ROC-AUC EER (%)
          Deepfakes            20        45.00  0.4900   47.50
          Face2Face            20        45.00  0.4975   47.50
           FaceSwap            20        55.00  0.5175   55.00
     NeuralTextures            20        50.00  0.5900   52.50


In [9]:
# ==============================================================================
# Cell 9: Video-Only Ablation Study (Paper Table 3) - FIXED
# ==============================================================================
def run_video_only_ablation(model, loader, device):
    model.eval()
    y_true = []
    y_scores_multimodal = []
    y_scores_ablation = []
    
    print("🔍 Running Audio-Ablation Study (Comparing Standard vs. Zeroed Audio Input)...")
    
    with torch.no_grad():
        for v_in, a_in, targets in loader:
            if v_in.numel() == 0:
                continue
            v_in = v_in.to(device)
            a_in_gpu = a_in.to(device)
            a_in_zero = torch.zeros_like(a_in_gpu)  # Zeroed out audio branch
            
            # 1. Standard Multimodal Inference
            logits_multi = model(v_in, a_in_gpu).squeeze(-1)
            probs_multi = torch.sigmoid(logits_multi)
            
            # 2. Ablated (Video-Only) Inference
            logits_ablation = model(v_in, a_in_zero).squeeze(-1)
            probs_ablation = torch.sigmoid(logits_ablation)
            
            y_true.extend(targets.cpu().numpy().flatten())
            y_scores_multimodal.extend(probs_multi.cpu().numpy().flatten())
            y_scores_ablation.extend(probs_ablation.cpu().numpy().flatten())
            
    y_true = np.array(y_true)
    y_scores_multimodal = np.array(y_scores_multimodal)
    y_scores_ablation = np.array(y_scores_ablation)
    
    # Calculate Multimodal Metrics
    acc_multi = accuracy_score(y_true, (y_scores_multimodal >= 0.5).astype(float)) * 100.0
    eer_multi, _, fpr_m, tpr_m = calculate_eer_safe(y_true, y_scores_multimodal)
    auc_multi = auc(fpr_m, tpr_m)
    
    # Calculate Ablated Metrics
    acc_abl = accuracy_score(y_true, (y_scores_ablation >= 0.5).astype(float)) * 100.0
    eer_abl, _, fpr_a, tpr_a = calculate_eer_safe(y_true, y_scores_ablation)
    auc_abl = auc(fpr_a, tpr_a)
    
    print("\n" + "=" * 70)
    print(" TABLE 3: MULTIMODAL VS. VIDEO-ONLY ABLATION COMPARISON")
    print("=" * 70)
    
    comp_df = pd.DataFrame([
        {
            'Pipeline Mode': 'Standard TSA-Net (Video + Audio)',
            'Accuracy (%)': f"{acc_multi:.2f}",
            'ROC-AUC': f"{auc_multi:.4f}",
            'EER (%)': f"{eer_multi * 100:.2f}"
        },
        {
            'Pipeline Mode': 'Video-Only Ablation (Audio Zeroed)',
            'Accuracy (%)': f"{acc_abl:.2f}",
            'ROC-AUC': f"{auc_abl:.4f}",
            'EER (%)': f"{eer_abl * 100:.2f}"
        }
    ])
    
    print(comp_df.to_string(index=False))
    print("=" * 70)

run_video_only_ablation(model, ffpp_loader, device)

🔍 Running Audio-Ablation Study (Comparing Standard vs. Zeroed Audio Input)...

 TABLE 3: MULTIMODAL VS. VIDEO-ONLY ABLATION COMPARISON
                     Pipeline Mode Accuracy (%) ROC-AUC EER (%)
  Standard TSA-Net (Video + Audio)        63.00  0.5238   49.38
Video-Only Ablation (Audio Zeroed)        63.00  0.5238   49.38
